# location coordinates

the web app has maps, so every location in the house and land data needs a latitude and longitude. i tried the kaggle sri lanka cities file first but a lot of its coordinates are wrong (ambalangoda ended up near negombo, alutgama near puttalam), so i geocode the names with openstreetmap instead.

In [1]:
import os, time, json, urllib.parse, urllib.request
import numpy as np
import pandas as pd

houses = pd.read_csv('../data/processed/houses_clean.csv')
land = pd.read_csv('../data/processed/land_clean.csv')
locs = (pd.concat([houses[['location', 'district']], land[['location', 'district']]])
        .groupby(['location', 'district']).size().rename('ads').reset_index())
print('locations:', len(locs))

locations: 204


the ad locations are towns and suburbs. each name is searched together with its district so places with the same name in other districts do not get mixed up. nominatim allows one request per second, so the first run takes a few minutes. results are saved to a csv and a later run only looks up the locations that are not in it yet (the land ads added two).

In [2]:
cache = '../data/processed/location_coords.csv'

def geocode(query):
    url = 'https://nominatim.openstreetmap.org/search?' + urllib.parse.urlencode(
        {'q': query, 'format': 'json', 'limit': 1, 'countrycodes': 'lk'})
    req = urllib.request.Request(url, headers={'User-Agent': 'lk-ensemble-lab-class-project'})
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.load(r)
    time.sleep(1.1)
    return (float(data[0]['lat']), float(data[0]['lon'])) if data else (np.nan, np.nan)

coords = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame(columns=['location', 'district', 'latitude', 'longitude'])
todo = locs[~locs['location'].isin(coords['location'])]
if len(todo):
    rows = []
    for _, r in todo.iterrows():
        name = r['location'].replace(' City', '')
        lat, lon = geocode(f"{name}, {r['district']} District, Sri Lanka")
        if np.isnan(lat):
            lat, lon = geocode(f"{name}, Sri Lanka")
        rows.append({'location': r['location'], 'district': r['district'], 'latitude': lat, 'longitude': lon})
    coords = pd.concat([coords, pd.DataFrame(rows)], ignore_index=True)
    coords.to_csv(cache, index=False)

print('not found:', coords['latitude'].isna().sum())

not found: 0


a geocoder can still return the wrong place, so i check how far each point is from the middle of its district. points that are far out get a closer look.

In [3]:
mid = coords.groupby('district')[['latitude', 'longitude']].transform('median')
coords['km_off'] = np.sqrt((coords['latitude'] - mid['latitude']) ** 2 + (coords['longitude'] - mid['longitude']) ** 2) * 111
coords.merge(locs[['location', 'ads']]).sort_values('km_off', ascending=False).head(12).round(2)

,location,district,latitude,longitude,km_off,ads
170,Puttalam City,Puttalam,8.03,79.83,68.67,86
66,Galgamuwa,Kurunegala,8.00,80.27,57.60,19
61,Embilipitiya,Ratnapura,6.33,80.85,56.04,233
10,Anamaduwa,Puttalam,7.88,80.01,54.90,35
127,Mahiyanganaya,Badulla,7.34,80.99,48.16,42
180,Sigiriya,Matale,7.96,80.76,45.82,42
26,Bibile,Monaragala,7.16,81.23,44.25,34
52,Deniyaya,Matara,6.34,80.56,40.79,18
104,Kataragama,Monaragala,6.41,81.33,39.83,48
202,Habarana,Anuradhapura,8.04,80.76,37.49,14


most of the far points are real, because some districts are just big (embilipitiya, kataragama, mahiyanganaya are all where they should be). one is wrong: alutgama came back inland near ratnapura, the real one is on the coast south of colombo. i fix that by hand, add akkaraipattu (spelled differently in the ads) and put the three places that were not found at the middle of their district.

In [4]:
fixes = {'Alutgama': (6.4337, 79.9993), 'Akkarepattu': (7.2184, 81.8541)}
for name, (lat, lon) in fixes.items():
    coords.loc[coords['location'] == name, ['latitude', 'longitude']] = [lat, lon]

middle = coords.groupby('district')[['latitude', 'longitude']].transform('median')
missing = coords['latitude'].isna()
coords.loc[missing, 'latitude'] = middle.loc[missing, 'latitude']
coords.loc[missing, 'longitude'] = middle.loc[missing, 'longitude']

coords = coords.drop(columns='km_off')
coords.to_csv(cache, index=False)
print('missing now:', coords['latitude'].isna().sum(), 'rows:', len(coords))

missing now: 0 rows: 204
